# Liver segmentation: protocol v1 launcher (Colab GPU)

This notebook only **calls the repository's own CLIs**. It contains no model, training or metric code, so the repository stays the single source of truth.

This trains a **new** model. It is a separate experiment from the historical result (*0.9886 Dice on the 26-volume 128³ centre-cropped validation split used for model selection*) and must never be reported in its place.

Rules (full text: `docs/EVALUATION_PROTOCOL.md`):
- 85 train / 20 validation / 26 test, seed 42, frozen in `splits/msd_task03_v1.json` and committed **before** training. The 26 test cases are drawn only from cases that were not the historical model's validation cases.
- Model selection and every development decision use the validation split only.
- The test split is evaluated **once**, for one frozen checkpoint, at the very end.
- Do not change anything after seeing test results. A lower score than the historical one is an acceptable outcome.

Status: written and checked locally with a synthetic dataset; not yet executed on Colab.

## 1. Settings (edit these)

In [ ]:
REPO_URL = "https://github.com/AdebanjiAdelowo/abdominal-ct-segmentation"
COMMIT = ""   # REQUIRED: the exact 40-character commit hash you reviewed. Never a branch name.
DRIVE = "/content/drive/MyDrive/liver_protocol_v1"          # persistent: checkpoints, logs, locks
MSD_TAR_ON_DRIVE = "/content/drive/MyDrive/msd/Task03_Liver.tar"   # or set MSD_DIR_ON_DRIVE below
MSD_DIR_ON_DRIVE = ""                                        # already-extracted Task03_Liver folder, if you have one
SEED = 0                                                     # training seed (the split seed is fixed at 42 in the repo)
CONFIRM_FINAL_TEST = False                                   # leave False until every modelling decision is final

import re
assert re.fullmatch(r"[0-9a-f]{40}", COMMIT), "Set COMMIT to a full 40-character hash so the run is traceable."

## 2. Persistent storage and repository at the pinned commit

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os; os.makedirs(DRIVE, exist_ok=True)

REPO = "/content/abdominal-ct-segmentation"
if not os.path.exists(REPO):
    !git clone {REPO_URL} {REPO}
%cd {REPO}
!git fetch --all -q && git checkout -q {COMMIT}
!git status --short --untracked-files=no && git log -1 --format='%H %s'
!pip install -q -r requirements.txt

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > GPU"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Dataset: original MSD Task03 NIfTI (copy to local disk, then validate)

In [ ]:
DATA = "/content/data/Task03_Liver"
if not os.path.isdir(DATA + "/imagesTr"):
    os.makedirs("/content/data", exist_ok=True)
    if MSD_DIR_ON_DRIVE:
        !cp -r "{MSD_DIR_ON_DRIVE}" /content/data/
    else:
        !tar -xf "{MSD_TAR_ON_DRIVE}" -C /content/data
!python -m src.data.validate_dataset --data-dir {DATA} --expect-cases 131

## 4. Frozen split

If `splits/msd_task03_v1.json` is already in the checked-out commit, it is verified and used. If not, it is generated **once**, saved to Drive, and this notebook stops: commit that file to the repository, take the new commit hash, and re-run from the top. Training refuses to start from an uncommitted split.

In [ ]:
SPLITS = f"{REPO}/splits/msd_task03_v1.json"
if os.path.exists(SPLITS):
    !python -m src.data.splits verify --splits {SPLITS}
    !python -m src.data.splits describe --data-dir {DATA} --splits {SPLITS}
else:
    !python -m src.data.splits make --data-dir {DATA} --out {SPLITS}
    !cp {SPLITS} {DRIVE}/msd_task03_v1.json
    raise SystemExit(f"Split generated and copied to {DRIVE}. Commit it as splits/msd_task03_v1.json, set COMMIT to the new hash, re-run.")

## 5. Smoke test (infrastructure only; its numbers are not results)

In [ ]:
!python -m src.smoke_test --data-dir {DATA} --splits {SPLITS} --out-dir {DRIVE}/smoke --mirror-dir {DRIVE}/smoke/persist

## 6. Training

Writes locally (fast) and mirrors verified copies of `best.pth`, `last.pth`, the manifest and the metrics log to Drive. If the runtime disconnects, run the resume cell.

In [ ]:
OUT = "/content/runs/protocol_v1"
PERSIST = f"{DRIVE}/runs/protocol_v1"
!python -m src.training.protocol_train --data-dir {DATA} --splits {SPLITS} --out-dir {OUT} --mirror-dir {PERSIST} --seed {SEED}

In [ ]:
# Resume after a disconnect: restore last.pth from Drive, then continue the same schedule.
os.makedirs(f"{OUT}/checkpoints", exist_ok=True)
!cp {PERSIST}/last.pth {OUT}/checkpoints/last.pth
!python -m src.training.protocol_train --data-dir {DATA} --splits {SPLITS} --out-dir {OUT} --mirror-dir {PERSIST} --seed {SEED} --resume {OUT}/checkpoints/last.pth

## 7. Validation evaluation of the selected checkpoint (full volumes, physical mm)

In [ ]:
EVAL = f"{DRIVE}/eval_protocol_v1"
!python -m src.inference.evaluate_protocol --checkpoint {PERSIST}/best.pth --data-dir {DATA} --splits {SPLITS} --split val --out-dir {EVAL}

## 8. FINAL test evaluation (once)

Only after the checkpoint is frozen. The evaluator requires the validation evaluation of this exact checkpoint above, requires the confirmation flag, and records the checkpoint in a lock file on Drive; a different checkpoint will be refused. The report lists complete failures (empty predictions) explicitly and never shows a finite-case HD95 without the failure count.

In [ ]:
assert CONFIRM_FINAL_TEST, "Set CONFIRM_FINAL_TEST = True only when the checkpoint is final."
LOCK = f"{DRIVE}/test_lock/msd_task03_v1.test_lock.json"
os.makedirs(os.path.dirname(LOCK), exist_ok=True)
!python -m src.inference.evaluate_protocol --checkpoint {PERSIST}/best.pth --data-dir {DATA} --splits {SPLITS} --split test --confirm-final-test --lock-file {LOCK} --out-dir {EVAL}

## 9. Afterwards

Download `test_summary.json`, `test_per_case.csv`, `val_summary.json`, `run_config.yaml` and the lock file from Drive. Report validation and test separately, label the model as new, and update the README and Selected Projects only from those files. Do not commit checkpoints or the dataset to Git.